# Comprehensibility Check

# Import packages

In [8]:
## use kernel normgrading (Python 3.11.9)

## connecting with server
import pandas as pd
import numpy as np
import sys
import os
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from tqdm import tqdm
import re
import json
from datetime import datetime
import requests

In [9]:
import openai
from openai._client import OpenAI
import dotenv

# Dataset

In [10]:
path = os.getcwd()
#print("Path to current file:", path)
df = pd.read_csv(path+"/data/rotexamples_07072025.csv")

In [11]:
df.head()

,situation-short-id,situation,rule-of-thumb (rot),rot-moral-foundations,rot-judgement,action,ID,MFT,stimuli-male,norm-target in stimuli-male,male,stimuli-female,norm-target in stimuli-female,gender,rot-agree,action-agree,action-pressure,action-hypothetical
0,reddit/amitheasshole/avxz68,not telling someone's girlfriend that he cheat...,It's good to speak up when someone's doing som...,care-harm,it is good,speaking up when someone's doing something har...,1,care-harm,William did not tell someone's girlfriend that...,William,male,Maria did not tell someone's girlfriend that h...,Maria,female,4,4,2,explicit-no
1,reddit/amitheasshole/alg2n1,rarely talking with my elderly roomates,It's good to create a friendly atmosphere when...,care-harm,it is good,creating a friendly atmosphere when you have r...,2,care-harm,John rarely talks to his elderly roommates and...,John,male,Mary rarely talks to her elderly roommates and...,Mary,female,3,3,1,explicit-no
2,reddit/amitheasshole/alv89u,getting upset/annoyed with my boyfriend for st...,It's inconsiderate to assume that your partner...,care-harm,it is inconsiderate,assumimg that your partner was not trying to s...,3,care-harm,Robert stayed up until 5 am because he assumed...,Robert,male,Barbara stayed up until 5 am because she assum...,Barbara,female,3,2,0,hypothetical
3,reddit/amitheasshole/a4l295,not listening to her about my health,You should take care of your health.,care-harm,should,taking care of your health.,4,care-harm,David does not listen to his girlfriend about ...,David,male,Jennifer does not listen to her boyfriend abou...,Jennifer,female,4,4,1,hypothetical
4,reddit/amitheasshole/atksbr,preventing my roommate from getting a new dog ...,You should grieve your dead pet before getting...,care-harm,you should,grieving your dead pet before getting a new on...,5,care-harm,Thomas did not grieve his dead dog before gett...,Thomas,male,Mary did not grieve her dead dog before gettin...,Mary,female,2,2,0,probable-no


# Prompt

In [15]:
## set model and temperature
def is_comprehensible(post, temp):
    keys = [
    "comprehensibility",
    "offensiveness",
    "prevalence",
    "subjectivity"
]

    system_prompt = """\
# System Prompt
You are an AI assistant trained to evaluate and annotate given social situation.
Rate each of the following questions on a scale of 1 (not at all) to 7 (very much).

# Criteria
"comprehensibility": "How easy is to understand the given social situation?",
"offensiveness": "How offensive is the given social situation?",
"prevalence": "How likely is the given social situation?",
"subjectivity": "How subjective is the given social situation? Consider actors' demographics, and situations. Will it it lead to different interpretations by different people?", 
"""

    json_schema = {
    "name": "annotation",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "comprehensibility": {
                "type": "integer",
                "description": "How easy is to understand the given social situation?",
                "enum": [1, 2, 3, 4, 5, 6, 7]
            },
            "offensiveness": {
                "type": "integer",
                "description": "How offensive is the given social situation?",
                "enum": [1, 2, 3, 4, 5, 6, 7]
            },
            "prevalence": {
                "type": "integer",
                "description": "How likely is the given social situation?",
                "enum": [1, 2, 3, 4, 5, 6, 7]
            },
            "subjectivity": {
                "type": "integer",
                "description": "How subjective is the given social situation? Consider actors' demographics, and situations. Will it lead to different interpretations by different people?",
                "enum": [1, 2, 3, 4, 5, 6, 7]
            }
        },
        "required": keys,
        "additionalProperties": False
    }
}

    response = openai.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": post
            }
        ],
        temperature=temp,
        max_tokens=2024,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0,
        response_format = {"type": "json_schema", "json_schema": json_schema}
      
    )

    output = response.choices[0].message.content
    # Parse the JSON output
    try:
        output = json.loads(output)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        output = None
    
    output_list = []
    if output is not None:
        output_list = [output[key] for key in keys if key in output]

    return output_list

# Calling LLM

In [16]:
## LLM API Calls - key
import dotenv

lm = "openai" 

Openai_KEY_FILE_LOCATION = '/Users/sr/Github/NormGrading/openai_key.env'
deepseek_KEY_FILE_LOCATION = '/Users/sr/Github/NormGrading/deepseek_key.env'

if lm == "openai":
    dotenv.load_dotenv(Openai_KEY_FILE_LOCATION)
    openai.api_key = os.environ["OPENAI_API_KEY"]
    model = "gpt-4o-mini"
    temp=0
elif lm=="deepseek":
    dotenv.load_dotenv(deepseek_KEY_FILE_LOCATION)
    deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
    API_URL = "https://api.deepseek.com/v1/chat/completions"
    temp=0

In [17]:
is_comprehensible("I am a student who is struggling with my studies. I feel overwhelmed and anxious about my exams.", temp)

[6, 1, 5, 4]

In [18]:
def get_annotations(lm, df, temp, message_column1,  message_id_column, output):
    """
    Processes a dataframe to extract metadiscourse features for each post.
    Uses a progress bar to display the annotation progress.
    """

    print("Extracting style features...")

    # Initialize progress bar
    with tqdm(total=len(df), desc="Processing Posts", unit=message_column1) as pbar:
        for index, row in df.iterrows():
            result = list()
            if lm=="openai": ##running openai
                result = is_comprehensible(row[message_column1], temp)

            ## Adding message_id to output -- add first feature name below
            result.append(row[message_id_column])
            result.append(row[message_column1])
            output.append(result)  # Save output incrementally
            #print(result)
            pbar.update(1)  # Update progress bar

    print("annotation completed.")

In [19]:
output =  list()
get_annotations("openai", df, 0, "stimuli-male", "situation-short-id", output)

Extracting style features...


Processing Posts: 100%|██████████| 451/451 [07:11<00:00,  1.04stimuli-male/s]

annotation completed.


In [20]:
keys = [
    "comprehensibility",
    "offensiveness",
    "prevalence",
    "subjectivity"
]

cols =  keys
cols.append("message_id")
cols.append("stimuli-male")
df_annotated = pd.DataFrame(output, columns=cols)

In [21]:
##saving 
timestamp = datetime.now().strftime('%Y%m%d')
output_path = f"/Users/sr/Github/NormGrading/output/{lm}_{timestamp}.csv"
print(f"Saving output - {output_path}")
df_annotated.to_csv(output_path, index=False)

Saving output - /Users/sr/Github/NormGrading/output/openai_20250711.csv
